# Avery Sovereign LoRA Fine-Tune
**Model:** Qwen/Qwen2-7B-Instruct → tastytator/avery-sovereign-lora  
**Dataset:** tastytator/sovereign-economy (sft/train — 16,150 pairs)  
**Hardware:** Kaggle T4 x1 (16 GB VRAM)  
**Method:** QLoRA (4-bit NF4) + SFTTrainer

Steps: Install → Load dataset → Load model (4-bit) → LoRA → Train → Push to HF


In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.44.2',
    'peft==0.12.0',
    'trl==0.9.6',
    'bitsandbytes==0.43.3',
    'datasets==2.21.0',
    'accelerate==0.33.0',
    'huggingface_hub',
    'sentencepiece',
], check=True)
print('Done.')

In [ ]:
# ── 2. Config ──────────────────────────────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')   # Add in Kaggle → Add-ons → Secrets

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_XET'] = '1'

BASE_MODEL   = 'Qwen/Qwen2-7B-Instruct'
DATASET_REPO = 'tastytator/sovereign-economy'
OUTPUT_DIR   = '/kaggle/working/avery-lora'
HF_PUSH_REPO = 'tastytator/avery-sovereign-lora'

LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
MAX_SEQ_LEN   = 1024
BATCH_SIZE    = 2
GRAD_ACCUM    = 8       # effective batch = 16
LR            = 2e-4
EPOCHS        = 2
WARMUP_RATIO  = 0.05
print(f'Config ready. Base: {BASE_MODEL}')

In [ ]:
# ── 3. Load and format dataset ─────────────────────────────────────────────
from datasets import load_dataset

ds = load_dataset(DATASET_REPO, name='sft', split='train', token=HF_TOKEN)
print(f'Loaded {len(ds)} pairs from sft/train')
print(ds[0])

In [ ]:
# ── 4. Format into chat template ───────────────────────────────────────────
SYSTEM_PROMPT = (
    'You are Avery, the sovereign business strategist for SovereignNation — '
    'a fixed-cost AI platform built for lower and middle class families, '
    'children education, and affordable connectivity. '
    'Use the KAIROS framework: Kickoff, Alignment, Implementation, Refinement, '
    'Optimization, Scaling. Be direct, structured, and actionable.'
)

def format_chat(example):
    messages = [
        {'role': 'system',    'content': SYSTEM_PROMPT},
        {'role': 'user',      'content': example['instruction']},
        {'role': 'assistant', 'content': example['response']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

print('Formatter ready — will apply after tokenizer loads')

In [ ]:
# ── 5. Load tokenizer ─────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print(f'Tokenizer loaded. Vocab: {tokenizer.vocab_size}')

In [ ]:
# ── 6. Format dataset ──────────────────────────────────────────────────────
ds_formatted = ds.map(format_chat, remove_columns=ds.column_names)

# Filter sequences that are too long
def is_short_enough(example):
    return len(tokenizer.encode(example['text'])) <= MAX_SEQ_LEN

ds_filtered = ds_formatted.filter(is_short_enough, num_proc=2)
print(f'After length filter: {len(ds_filtered)}/{len(ds_formatted)} examples kept')
print(ds_filtered[0]['text'][:300])

In [ ]:
# ── 7. Load model in 4-bit (QLoRA) ────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

In [ ]:
# ── 8. LoRA config ────────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 9. Training arguments ─────────────────────────────────────────────────
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    save_steps=100,
    logging_steps=25,
    learning_rate=LR,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type='cosine',
    report_to='none',
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_filtered,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    tokenizer=tokenizer,
    packing=True,
)

print(f'Trainer ready. Steps per epoch: {len(trainer.get_train_dataloader())}')

In [ ]:
# ── 10. TRAIN ─────────────────────────────────────────────────────────────
print('Starting training...')
trainer.train()
print('Training complete.')

In [ ]:
# ── 11. Save locally ──────────────────────────────────────────────────────
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Saved to {OUTPUT_DIR}')

In [ ]:
# ── 12. Push LoRA adapter to HuggingFace ──────────────────────────────────
from huggingface_hub import login
login(token=HF_TOKEN)

trainer.model.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_PUSH_REPO, token=HF_TOKEN)
print(f'Pushed to https://huggingface.co/{HF_PUSH_REPO}')

In [ ]:
# ── 13. Quick inference test ──────────────────────────────────────────────
from peft import PeftModel
from transformers import pipeline

# Merge for inference
merged = trainer.model.merge_and_unload()

pipe = pipeline('text-generation', model=merged, tokenizer=tokenizer,
                device_map='auto', max_new_tokens=300)

test_prompt = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'Build a go-to-market strategy for SovereignNation targeting rural communities.'},
]

result = pipe(tokenizer.apply_chat_template(test_prompt, tokenize=False, add_generation_prompt=True),
              temperature=0.7, do_sample=True)
print(result[0]['generated_text'][-800:])